# 13 (Mac) - Louvain split, full comparison

The Louvain split was only tested with FedAvg vs topo (notebook 11), and hospitals were matched by
index across seeds. This notebook fixes both:

- **Louvain's hospital count is set by the graph, not by `n_clients`.** Expect 3 hospitals for every seed.
- **Hospitals are sorted by size within each seed** (largest / middle / smallest), so matching is by
  rank, not by index. Seed 44 swaps the two smaller hospitals relative to the others.
- **Every arm is included:** local-only, pooled, FedAvg, FedProx, FedMedian, FedTrimmedAvg,
  uniform-weight control, and topo (T=1). Same protocol as notebook 12, five seeds.
- **Topo weights are recomputed** from the fingerprints for each seed, not read from logs.

Results go to `results/amr_louvain_full.json` (repo root). It resumes from that file if it exists.


In [ ]:
# 1) Environment
import sys, torch, ray, flwr, torch_geometric
print('python', sys.version.split()[0], '| torch', torch.__version__, '| flwr', flwr.__version__,
      '| ray', ray.__version__, '| torch_geometric', torch_geometric.__version__)
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
# 2) Point at the local ARMD folder
import os
os.environ.setdefault('ARMD_DIR', '/Users/vikramadityasharma/Library/CloudStorage/GoogleDrive-vikramaditya.raunak@gmail.com/.shortcut-targets-by-id/1uVx5PmldPyJBMtnB9Lkl9N1rNBQrmJt-/ARMD')  # edit if your folder differs
print('ARMD_DIR:', os.environ['ARMD_DIR'])


In [ ]:
# 3) Sanity: data resolves + imports
import sys
from pathlib import Path
# find the repo's src/ folder whether the notebook runs from notebooks/ or the repo root
SRC = next(p / 'src' for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'amr_fed').is_dir())
sys.path.insert(0, str(SRC))
from amr_fed import config
D = Path(config.DATA_DIR)
print('DATA_DIR:', D, '| exists:', D.exists())
assert D.exists(), 'ARMD_DIR is wrong - fix cell 2 and re-run.'
from amr_fed.federated.run import run_multiseed
from amr_fed.data_loader import load_cohort_frame, PK
from amr_fed.partition import louvain_split, partition_summary
from amr_fed.topology import compute_fingerprint, aggregation_weights
print('imports OK')


In [ ]:
# 4) Load cohort once
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.data_loader import load_cohort_frame
df = load_cohort_frame()
print('rows:', f'{len(df):,}')


In [ ]:
# 5) Smoke: what Louvain actually produces for each seed (expect 3 hospitals)
from amr_fed.partition import louvain_split
SEEDS = (42, 43, 44, 45, 46)
for s in SEEDS:
    a = louvain_split(df, n_clients=3, seed=s)
    sizes = a.value_counts().sort_values(ascending=False)
    print(f'seed {s}: {a.nunique()} hospitals | patients per hospital (largest first): {sizes.tolist()}')
    assert a.index.is_unique and len(a) == df[PK].nunique(), 'not 1:1'
print('SMOKE PASS')


In [ ]:
# 6) All arms, identical protocol. Resumes from results/amr_louvain_full.json.
import os, ray, json
from pathlib import Path
os.environ.pop('AMR_RNG_PROBE', None)
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.federated.run import run_multiseed
from amr_fed.partition import louvain_split

KW = dict(seeds=SEEDS, df=df, partition_fn=lambda d, s: louvain_split(d, n_clients=3, seed=s),
          n_clients=3, rounds=8, local_epochs=4, local_only_epochs=32, hidden=128)
ARMS = [  # (key, label, extra kwargs)
    ('fedavg',        'louvain/FedAvg',        dict(strategy='fedavg')),
    ('fedprox',       'louvain/FedProx',       dict(strategy='fedprox')),
    ('fedmedian',     'louvain/FedMedian',     dict(strategy='fedmedian')),
    ('fedtrimmedavg', 'louvain/FedTrimmedAvg', dict(strategy='fedtrimmedavg')),
    ('uniform',       'louvain/UNIFORM (T=inf control)', dict(strategy='topology', topo_mode='distinctiveness', topo_temperature=float('inf'))),
    ('topo',          'louvain/TOPO (T=1)',    dict(strategy='topology', topo_mode='distinctiveness', topo_temperature=1.0)),
]
RESULTS = SRC.parent / 'results' / 'amr_louvain_full.json'
RESULTS.parent.mkdir(exist_ok=True)
res = json.load(open(RESULTS)) if RESULTS.exists() else {}
for i, (key, label, extra) in enumerate(ARMS):
    if key in res:
        print(f'skip {key}: already saved')
        continue
    ray.shutdown()
    res[key] = run_multiseed(label=label, compute_pooled=(i == 0), **KW, **extra)
    json.dump(res, open(RESULTS, 'w'), default=str)  # checkpoint after every arm
    print(f'checkpoint saved after {key}')


In [ ]:
# 7) Per-hospital comparison, hospitals ranked by size within each seed
import numpy as np
names = {'fedavg':'FedAvg','fedprox':'FedProx','fedmedian':'FedMed','fedtrimmedavg':'FedTrim','uniform':'Uniform','topo':'Topo'}
keys = [k for k, _, _ in ARMS]
ref = res['fedavg']['runs']

def ranked(i):
    """Client ids for seed i, largest hospital first."""
    sizes = ref[i]['sizes']
    return sorted(range(len(sizes)), key=lambda c: -sizes[c])

for rank, tag in enumerate(['largest', 'middle', 'smallest']):
    print(f'\n=== {tag} hospital (mean over seeds, F1 at best round) ===')
    print(f"{'method':<10}{'local':>8}{'pooled':>8}" + ''.join(f'{names[k]:>9}' for k in keys))
    lo = [ref[i]['local_only_per_client'][ranked(i)[rank]] for i in range(len(SEEDS))]
    po = [ref[i]['pooled_per_client'][ranked(i)[rank]] for i in range(len(SEEDS))]
    row = {k: [res[k]['runs'][i]['fedavg_per_client'][ranked(i)[rank]] for i in range(len(SEEDS))] for k in keys}
    print(f"{'':<10}{np.mean(lo):>8.4f}{np.mean(po):>8.4f}" + ''.join(f'{np.mean(row[k]):>9.4f}' for k in keys))
    print(f"{'  n_pat':<10}{np.mean([ref[i]['sizes'][ranked(i)[rank]] for i in range(len(SEEDS))]):>8.0f}")

print('\n=== summary (mean +/- std over seeds) ===')
def f(ms): return 'n/a' if ms is None or ms[0] is None else f'{ms[0]:.4f} +/- {ms[1]:.4f}'
print(f"{'method':<12} {'best-round':<18} {'final-round':<18} {'worst-hospital':<18} {'AUROC':<18}")
print(f"{'local-only':<12} {f(res['fedavg']['local_only']):<18} {'-':<18} {f(res['fedavg']['worst_local']):<18} {f(res['fedavg']['local_only_auroc']):<18}")
print(f"{'pooled':<12} {f(res['fedavg']['pooled']):<18} {'-':<18} {f(res['fedavg']['pooled_worst']):<18} {f(res['fedavg']['pooled_auroc']):<18}")
for k in keys:
    r = res[k]
    print(f"{names[k]:<12} {f(r['fedavg_best']):<18} {f(r['fedavg_final']):<18} {f(r['worst_fed']):<18} {f(r['fedavg_auroc_best']):<18}")


In [ ]:
# 8) Topo weights recomputed from fingerprints, per seed, hospitals largest first
from amr_fed.partition import louvain_split
for s in SEEDS:
    a = louvain_split(df, n_clients=3, seed=s)
    ids = a.value_counts().sort_values(ascending=False).index.tolist()  # client ids, largest first
    subs = [df[df[PK].map(a) == c] for c in ids]
    fps = [compute_fingerprint(sub) for sub in subs]
    sizes = [sub[PK].nunique() for sub in subs]
    tot = sum(sizes)
    w = aggregation_weights(fps, temperature=1.0, mode='distinctiveness')
    print(f'seed {s}:')
    for r, (n, wt) in enumerate(zip(sizes, w)):
        print(f'   rank {r}: patients={n:>6}  size-share={n/tot:.3f}  topo weight={wt:.3f}  uniform=={1/len(w):.3f}')
